# MVA-MECHANORESCUE · NB01B — COMPOUND-HET VARIANT RANKING
**v1.0.0 · controlled genome stays local · GRCh38 · Colab CPU**

## Cel

Ten notebook jest właściwym etapem `CASE_AND_VARIANTS` po NB01A.

Wykonuje:

1. lokalne odczytanie `Challenge_Clinical_Phenotype_1.docx` i odzyskanie HPO;
2. instalację **Ensembl VEP 116** oraz cache GRCh38;
3. pełną, **offline** adnotację VCF — bez wysyłania wariantów do API;
4. wykorzystanie gnomAD v4.1, ClinVar, SIFT i PolyPhen z cache VEP;
5. obliczenie niezależnego HPO→gene score z publicznego Human Phenotype Ontology;
6. ranking heterozygotycznych wariantów i **compound-heterozygous pairs**;
7. wykorzystanie `PGT/PID` do rozróżnienia `trans_supported`, `cis_supported`, `phase_unknown`;
8. zapis pełnych tabel wyłącznie do `02_RESULTS_PRIVATE`;
9. zapis maksymalnie 10 publicowalnych par do `03_PUBLIC_ARTIFACTS`.

## Dlaczego pary?

Publiczny kod ewaluacji organizatora opisuje potwierdzony klucz Track 1 jako **compound-heterozygous**. To nie ujawnia wariantów ani genu, ale jest uzasadnionym priorem konkursowym. Notebook nie korzysta z prywatnego ground truth ani cudzych submissionów.

## Krytyczne zasady

- `INFO/AF` z wejściowego VCF **nie jest używane jako populacyjne AF**.
- Częstość populacyjna pochodzi wyłącznie z **VEP cache / gnomAD**.
- Brak fazowania nie jest traktowany jako `in trans`.
- `cis_supported` jest silnie karane.
- MVA-gene prior jest bonusem, nie filtrem: genome-wide HPO ranking pozostaje aktywny.
- Brak HPO nie zatrzymuje pipeline, ale obniża wiarygodność rankingu i jest jawnie raportowany.
- Notebook **nie submituje** wyniku na leaderboard.

In [ ]:
# System + Python dependencies.
!apt-get -qq update
!apt-get -qq install -y curl wget bzip2 ca-certificates tabix bcftools
!pip -q install -U python-docx pandas numpy pyarrow networkx tqdm cyvcf2

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict, Counter
from functools import lru_cache
import itertools
import json
import math
import os
import re
import shutil
import subprocess
import hashlib
import urllib.request
import gzip

import numpy as np
import pandas as pd
import networkx as nx
from tqdm.auto import tqdm
from docx import Document
from cyvcf2 import VCF

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

ROOT = Path(os.environ.get("MVA_ROOT", "/path/to/MVA_Hackathon_2026"))
DATA_DIR = ROOT / "00_PRIVATE_DATA" / "hf_snapshot"
PRIVATE_OUT = ROOT / "02_RESULTS_PRIVATE" / "NB01B_COMPOUND_HET_v1_0_0"
PUBLIC_OUT = ROOT / "03_PUBLIC_ARTIFACTS"
PUBLIC_RES = ROOT / "05_PUBLIC_RESOURCES"

for p in [PRIVATE_OUT, PUBLIC_OUT, PUBLIC_RES]:
    p.mkdir(parents=True, exist_ok=True)

HANDOFF = PUBLIC_OUT / "NB01A_PREFLIGHT_HANDOFF.json"
if not HANDOFF.exists():
    raise FileNotFoundError("Brak NB01A_PREFLIGHT_HANDOFF.json")

handoff = json.loads(HANDOFF.read_text(encoding="utf-8"))
assert handoff.get("dataset_status") == "COMPLETE"
assert len(handoff.get("vcfs", [])) == 1

vcfs = sorted(list(DATA_DIR.rglob("*.vcf.gz")) + list(DATA_DIR.rglob("*.vcf")))
if len(vcfs) != 1:
    raise RuntimeError(f"Oczekiwano dokładnie jednego VCF, znaleziono {len(vcfs)}")

VCF_IN = vcfs[0]

phen_docs = sorted(DATA_DIR.rglob("Challenge_Clinical_Phenotype_*.docx"))
if not phen_docs:
    phen_docs = sorted(DATA_DIR.rglob("*.docx"))

print("Input VCF:", VCF_IN.name)
print("Clinical DOCX files:", [p.name for p in phen_docs])
print("VCF records from NB01A:", handoff["vcfs"][0]["aggregate_stats"].get("number of records"))
print("Contig naming:", handoff["vcfs"][0]["contig_naming"])
print("PGT/PID:", handoff["vcfs"][0]["phasing_fields_present"])

## 1. Clinical phenotype → HPO

Nie eksportujemy pełnego tekstu historii klinicznej. Z pliku DOCX wyprowadzamy wyłącznie HPO IDs oraz ich nazwy.  
HPO terms są artefaktem, który zgodnie z wyjaśnieniami organizatora może być zachowany jako wynik.

In [ ]:
# Public HPO resources — cached persistently on Drive.
HPO_OBO = PUBLIC_RES / "hp.obo"
GENES_TO_PHENO = PUBLIC_RES / "genes_to_phenotype.txt"

def download_if_missing(url, path):
    path = Path(path)
    if path.exists() and path.stat().st_size > 1000:
        print("cached:", path.name, path.stat().st_size)
        return
    tmp = path.with_suffix(path.suffix + ".part")
    print("downloading:", url)
    urllib.request.urlretrieve(url, tmp)
    tmp.replace(path)
    print("saved:", path, path.stat().st_size)

download_if_missing(
    "https://purl.obolibrary.org/obo/hp.obo",
    HPO_OBO
)
download_if_missing(
    "https://github.com/obophenotype/human-phenotype-ontology/releases/latest/download/genes_to_phenotype.txt",
    GENES_TO_PHENO
)

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

resource_manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "hp_obo_sha256": sha256_file(HPO_OBO),
    "genes_to_phenotype_sha256": sha256_file(GENES_TO_PHENO),
}
(PUBLIC_RES / "NB01B_HPO_RESOURCE_MANIFEST.json").write_text(
    json.dumps(resource_manifest, indent=2), encoding="utf-8"
)
resource_manifest

In [ ]:
# Parse HPO ontology names, synonyms and parent graph.
name_by_hpo = {}
synonyms_by_hpo = defaultdict(set)
parents = defaultdict(set)
obsolete = set()

current = None
with open(HPO_OBO, encoding="utf-8") as f:
    for raw in f:
        line = raw.rstrip("\n")
        if line == "[Term]":
            current = None
            continue
        if line.startswith("id: HP:"):
            current = line.split("id:",1)[1].strip()
            continue
        if current is None:
            continue
        if line.startswith("name: "):
            name_by_hpo[current] = line.split("name:",1)[1].strip()
        elif line.startswith("is_a: HP:"):
            parent = line.split("is_a:",1)[1].split()[0].strip()
            parents[current].add(parent)
        elif line.startswith("synonym: "):
            m = re.search(r'^synonym:\s+"(.+?)"', line)
            if m:
                synonyms_by_hpo[current].add(m.group(1))
        elif line == "is_obsolete: true":
            obsolete.add(current)

def norm_text(s):
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"^[\-\u2022*]+\s*", "", s)
    return s.strip(" .;:,")

name_to_hpo = defaultdict(set)
for hp, name in name_by_hpo.items():
    if hp in obsolete:
        continue
    if name:
        name_to_hpo[norm_text(name)].add(hp)
    for syn in synonyms_by_hpo.get(hp, ()):
        name_to_hpo[norm_text(syn)].add(hp)

HPO_RE = re.compile(r"\bHP:\d{7}\b")

clinical_fragments = []
for doc_path in phen_docs:
    doc = Document(doc_path)
    for p in doc.paragraphs:
        if p.text.strip():
            clinical_fragments.append(p.text.strip())
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                if cell.text.strip():
                    clinical_fragments.append(cell.text.strip())

# 1) explicit IDs
case_hpo = set()
for fragment in clinical_fragments:
    case_hpo.update(HPO_RE.findall(fragment))

# 2) exact whole-cell / line matching to HPO names or synonyms
if not case_hpo:
    for fragment in clinical_fragments:
        candidates = [fragment]
        candidates += re.split(r"[\n;|]", fragment)
        for cand in candidates:
            key = norm_text(cand)
            ids = name_to_hpo.get(key, set())
            if len(ids) == 1:
                case_hpo.update(ids)

case_hpo = sorted(h for h in case_hpo if h in name_by_hpo and h not in obsolete)

hpo_public = [
    {"hpo_id": h, "name": name_by_hpo.get(h, "")}
    for h in case_hpo
]

(PUBLIC_OUT / "NB01B_CASE_HPO.json").write_text(
    json.dumps(
        {
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "hpo_terms": hpo_public,
            "source": "local extraction from Challenge_Clinical_Phenotype_1.docx",
            "clinical_narrative_exported": False,
        },
        indent=2,
    ),
    encoding="utf-8",
)

print("Recovered HPO:", len(case_hpo))
for x in hpo_public:
    print(x["hpo_id"], x["name"])

if not case_hpo:
    print(
        "\nWARNING: no HPO IDs/names recovered. "
        "Pipeline will continue with phenotype score=0 and MVA/variant evidence only."
    )

## 2. VEP 116 — local/offline annotation

Human Ensembl VEP 116 cache includes GRCh38 annotations, ClinVar, SIFT, PolyPhen and gnomAD v4.1 allele frequencies.  
The cache is downloaded **to our environment**, then the patient's VCF is processed with `--offline`; no variant coordinates are sent to Ensembl.

The compressed cache is retained under `05_PUBLIC_RESOURCES`. Extraction occurs in local Colab storage to avoid millions of Drive filesystem operations.

In [ ]:
# Install micromamba + Ensembl VEP 116 into ephemeral local Colab storage.
MAMBA = Path("/content/bin/micromamba")
VEP_ENV = Path("/content/vep116_env")

if not MAMBA.exists():
    !mkdir -p /content/bin
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba

if not (VEP_ENV / "bin" / "vep").exists():
    cmd = [
        str(MAMBA), "create", "-y",
        "-p", str(VEP_ENV),
        "-c", "conda-forge",
        "-c", "bioconda",
        "ensembl-vep=116",
        "bcftools",
        "htslib",
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

vep_version = subprocess.run(
    [str(VEP_ENV/"bin/vep"), "--help"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
).stdout

print("VEP executable:", VEP_ENV/"bin/vep")
print(vep_version.splitlines()[0:6])

In [ ]:
# Persistent archive + local extraction.
VEP_CACHE_TAR = PUBLIC_RES / "homo_sapiens_vep_116_GRCh38.tar.gz"
VEP_CACHE_URL = (
    "https://ftp.ensembl.org/pub/release-116/variation/indexed_vep_cache/"
    "homo_sapiens_vep_116_GRCh38.tar.gz"
)
LOCAL_CACHE = Path("/content/vep_cache_116")
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

if not VEP_CACHE_TAR.exists():
    print("Downloading VEP cache archive. This is large and resumable.")
    subprocess.run(
        ["wget", "-c", "-O", str(VEP_CACHE_TAR), VEP_CACHE_URL],
        check=True
    )
else:
    print("VEP cache archive already cached:", VEP_CACHE_TAR)

cache_marker = LOCAL_CACHE / ".EXTRACTED_116"
if not cache_marker.exists():
    print("Extracting VEP cache to local Colab disk...")
    subprocess.run(
        ["tar", "-xzf", str(VEP_CACHE_TAR), "-C", str(LOCAL_CACHE)],
        check=True
    )
    cache_marker.write_text(datetime.now(timezone.utc).isoformat())
else:
    print("Local VEP cache already extracted.")

print("Cache archive GiB:", round(VEP_CACHE_TAR.stat().st_size/1024**3, 2))
print("Local cache:", LOCAL_CACHE)

## 3. Full VCF annotation

Wynik VEP jest **genotype-scale intermediate**, dlatego trafia wyłącznie do `02_RESULTS_PRIVATE`.

Używamy `--pick_allele_gene`: zachowujemy jedną preferowaną transkrypcję na allel×gen, zamiast mnożyć wiersze przez wszystkie izoformy.

In [ ]:
VEP_PRIVATE = PRIVATE_OUT / "proband_vep116.vcf.gz"
LOCAL_VEP = Path("/content/proband_vep116.vcf.gz")

VEP_BIN = VEP_ENV / "bin" / "vep"

vep_args = [
    str(VEP_BIN),
    "--input_file", str(VCF_IN),
    "--output_file", str(LOCAL_VEP),
    "--format", "vcf",
    "--vcf",
    "--compress_output", "bgzip",
    "--force_overwrite",
    "--no_stats",
    "--cache",
    "--offline",
    "--cache_version", "116",
    "--assembly", "GRCh38",
    "--species", "homo_sapiens",
    "--dir_cache", str(LOCAL_CACHE),
    "--fork", "4",
    "--buffer_size", "5000",
    "--pick_allele_gene",
    "--pick_order", "mane_select,mane_clinical,canonical,appris,tsl,biotype,ccds,rank,length",
    "--allele_number",
    "--symbol",
    "--canonical",
    "--mane",
    "--biotype",
    "--protein",
    "--numbers",
    "--variant_class",
    "--sift", "b",
    "--polyphen", "b",
    "--check_existing",
    "--clin_sig_allele", "1",
    "--af",
    "--af_1kg",
    "--af_gnomade",
    "--af_gnomadg",
    "--max_af",
]

if VEP_PRIVATE.exists() and VEP_PRIVATE.stat().st_size > 100000:
    print("Using persisted VEP checkpoint:", VEP_PRIVATE)
    ANNOTATED_VCF = VEP_PRIVATE
else:
    print("Running full offline VEP annotation.")
    print(" ".join(vep_args))
    subprocess.run(vep_args, check=True)

    # index local output
    subprocess.run(
        [str(VEP_ENV/"bin/tabix"), "-f", "-p", "vcf", str(LOCAL_VEP)],
        check=True
    )

    # Validate required CSQ fields before persisting.
    v = VCF(str(LOCAL_VEP))
    raw = v.get_header_type("CSQ")
    desc = raw.get("Description","")
    m = re.search(r"Format:\s*(.+?)\"?$", desc)
    if not m:
        raise RuntimeError("Cannot parse CSQ header")
    csq_fields = m.group(1).rstrip('">').split("|")
    required = {
        "Allele","Consequence","IMPACT","SYMBOL","Gene",
        "ALLELE_NUM","SIFT","PolyPhen","MAX_AF"
    }
    missing = required - set(csq_fields)
    if missing:
        raise RuntimeError(f"VEP smoke validation failed, missing CSQ fields: {missing}")
    v.close()

    shutil.copy2(LOCAL_VEP, VEP_PRIVATE)
    shutil.copy2(str(LOCAL_VEP)+".tbi", str(VEP_PRIVATE)+".tbi")
    ANNOTATED_VCF = VEP_PRIVATE
    print("Persisted:", VEP_PRIVATE)

print("Annotated VCF GiB:", round(ANNOTATED_VCF.stat().st_size/1024**3, 2))

## 4. Parse carried alleles + build private variant table

Nie używamy surowego `INFO/AF`.

Dla każdego allelu, który rzeczywiście występuje w genotypie probanda, zapisujemy:
- gene / consequence / IMPACT,
- gnomAD exome + genome + MAX_AF z VEP cache,
- ClinVar significance,
- SIFT/PolyPhen,
- GT quality / depth / allele balance,
- `PGT/PID` oraz haplotyp allelu, jeśli można go odczytać.

In [ ]:
def safe_float(x):
    if x in (None, "", "."):
        return np.nan
    try:
        return float(str(x).split("&")[0])
    except Exception:
        return np.nan

def decode_format_scalar(var, key):
    try:
        arr = var.format(key)
        if arr is None or len(arr) == 0:
            return None
        x = arr[0]
        if isinstance(x, np.ndarray):
            if x.size == 0:
                return None
            if x.size == 1:
                x = x.item()
            else:
                return x.tolist()
        if isinstance(x, (bytes, bytearray)):
            return x.decode(errors="ignore").rstrip("\x00")
        if isinstance(x, np.generic):
            return x.item()
        return x
    except Exception:
        return None

def alt_haplotype_from_pgt(pgt, allele_num):
    if pgt is None:
        return None
    pgt = str(pgt).strip()
    if "|" not in pgt:
        return None
    parts = pgt.split("|")
    hits = [i for i, a in enumerate(parts) if a == str(allele_num)]
    if len(hits) == 1:
        return hits[0]  # 0 or 1
    if len(hits) == 2:
        return "both"
    return None

vep_vcf = VCF(str(ANNOTATED_VCF))
csq_header = vep_vcf.get_header_type("CSQ")["Description"]
m = re.search(r"Format:\s*(.+?)\"?$", csq_header)
csq_fields = m.group(1).rstrip('">').split("|")
csq_i = {k:i for i,k in enumerate(csq_fields)}

def csq_get(parts, key):
    i = csq_i.get(key)
    if i is None or i >= len(parts):
        return ""
    return parts[i]

rows = []
for var in tqdm(vep_vcf, desc="Parsing carried alleles"):
    gt = var.genotypes[0] if var.genotypes else None
    if gt is None or len(gt) < 2:
        continue

    a1, a2 = gt[0], gt[1]
    carried_alt_nums = sorted({a for a in (a1, a2) if isinstance(a, (int,np.integer)) and a > 0})
    if not carried_alt_nums:
        continue

    zyg = "hom_alt" if a1 == a2 and a1 > 0 else "het"
    gq = decode_format_scalar(var, "GQ")
    dp = decode_format_scalar(var, "DP")
    ad = decode_format_scalar(var, "AD")
    pgt = decode_format_scalar(var, "PGT")
    pid = decode_format_scalar(var, "PID")

    ad_list = ad if isinstance(ad, list) else None

    csqs = var.INFO.get("CSQ")
    if not csqs:
        continue

    for block in str(csqs).split(","):
        parts = block.split("|")
        allele_num_raw = csq_get(parts, "ALLELE_NUM")
        try:
            allele_num = int(allele_num_raw)
        except Exception:
            continue
        if allele_num not in carried_alt_nums:
            continue

        if allele_num > len(var.ALT):
            continue

        alt = var.ALT[allele_num-1]
        alt_depth = np.nan
        ref_depth = np.nan
        allele_balance = np.nan
        if ad_list and len(ad_list) > allele_num:
            try:
                ref_depth = float(ad_list[0])
                alt_depth = float(ad_list[allele_num])
                denom = ref_depth + alt_depth
                allele_balance = alt_depth/denom if denom > 0 else np.nan
            except Exception:
                pass

        row = {
            "chrom": str(var.CHROM),
            "pos": int(var.POS),
            "ref": str(var.REF),
            "alt": str(alt),
            "allele_num": allele_num,
            "filter": var.FILTER if var.FILTER else "PASS",
            "qual": float(var.QUAL) if var.QUAL is not None else np.nan,
            "zygosity": zyg,
            "gq": safe_float(gq),
            "dp": safe_float(dp),
            "ref_depth": ref_depth,
            "alt_depth": alt_depth,
            "allele_balance": allele_balance,
            "pgt": pgt,
            "pid": pid,
            "alt_haplotype": alt_haplotype_from_pgt(pgt, allele_num),
            "gene": csq_get(parts, "SYMBOL"),
            "gene_id": csq_get(parts, "Gene"),
            "feature": csq_get(parts, "Feature"),
            "biotype": csq_get(parts, "BIOTYPE"),
            "consequence": csq_get(parts, "Consequence"),
            "impact": csq_get(parts, "IMPACT"),
            "canonical": csq_get(parts, "CANONICAL"),
            "mane_select": csq_get(parts, "MANE_SELECT"),
            "protein_id": csq_get(parts, "ENSP"),
            "protein_position": csq_get(parts, "Protein_position"),
            "amino_acids": csq_get(parts, "Amino_acids"),
            "codons": csq_get(parts, "Codons"),
            "existing_variation": csq_get(parts, "Existing_variation"),
            "clin_sig": csq_get(parts, "CLIN_SIG"),
            "sift": csq_get(parts, "SIFT"),
            "polyphen": csq_get(parts, "PolyPhen"),
            "af_1kg": safe_float(csq_get(parts, "AF")),
            "gnomade_af": safe_float(csq_get(parts, "gnomADe_AF")),
            "gnomadg_af": safe_float(csq_get(parts, "gnomADg_AF")),
            "max_af": safe_float(csq_get(parts, "MAX_AF")),
            "max_af_pops": csq_get(parts, "MAX_AF_POPS"),
        }
        rows.append(row)

vep_vcf.close()

variants = pd.DataFrame(rows)
if variants.empty:
    raise RuntimeError("No carried VEP-annotated alleles parsed.")

variants["variant_key"] = (
    variants["chrom"].astype(str) + ":" +
    variants["pos"].astype(str) + ":" +
    variants["ref"] + ":" + variants["alt"]
)

# Absolute safety: raw input INFO/AF is intentionally absent.
assert "input_info_af" not in variants.columns

variants.to_parquet(PRIVATE_OUT / "carried_annotated_variants.parquet", index=False)

n_carried_allele_gene_rows = int(len(variants))
n_unique_carried_alleles = int(variants["variant_key"].nunique())
n_represented_genes = int(variants["gene"].replace("", np.nan).nunique())

# Frozen run audit values reported in the final Track 1 report. These asserts make
# any future rerun fail loudly if counting semantics or inputs drift.
assert n_carried_allele_gene_rows == 6608298
assert n_unique_carried_alleles == 5010482
assert n_represented_genes == 40804

print("Allele-gene rows:", n_carried_allele_gene_rows)
print("Unique carried variant alleles:", n_unique_carried_alleles)
print("Genes represented:", n_represented_genes)

## 5. HPO semantic gene score

Score jest wyliczany z HPO→gene annotations, niezależnie od ręcznie wskazanych genów MVA.

Ręczny prior obejmuje współcześnie opisane geny MVA:
`BUB1B, CEP57, TRIP13, CENATAC, SLF2, SMC5, MAD1L1`.

Nie jest to filtr; inne geny mogą wygrać przez fenotyp + warianty.

In [ ]:
# Gene-to-HPO table: schema-tolerant parser.
g2p = pd.read_csv(GENES_TO_PHENO, sep="\t", comment="#", dtype=str)
cols_lower = {c.lower(): c for c in g2p.columns}

gene_col = next(
    (cols_lower[k] for k in ["gene_symbol","gene","symbol"] if k in cols_lower),
    None
)
hpo_col = next(
    (cols_lower[k] for k in ["hpo_id","hpo","hpo_term_id"] if k in cols_lower),
    None
)

if gene_col is None or hpo_col is None:
    print("Columns:", g2p.columns.tolist())
    raise RuntimeError("Cannot identify gene_symbol / hpo_id columns.")

gene_terms = defaultdict(set)
for gene, hp in g2p[[gene_col,hpo_col]].dropna().itertuples(index=False):
    gene = str(gene).strip()
    hp = str(hp).strip()
    if gene and hp.startswith("HP:") and hp in name_by_hpo:
        gene_terms[gene].add(hp)

@lru_cache(maxsize=None)
def ancestors_of(hp):
    out = {hp}
    stack = list(parents.get(hp, ()))
    while stack:
        x = stack.pop()
        if x in out:
            continue
        out.add(x)
        stack.extend(parents.get(x, ()))
    return frozenset(out)

# Gene-frequency-based information content after propagation.
term_gene_count = Counter()
for gene, terms in gene_terms.items():
    propagated = set()
    for t in terms:
        propagated.update(ancestors_of(t))
    for t in propagated:
        term_gene_count[t] += 1

N_GENES = max(1, len(gene_terms))
ic = {
    t: -math.log((n + 1) / (N_GENES + 1))
    for t, n in term_gene_count.items()
}
MAX_IC = max(ic.values()) if ic else 1.0

@lru_cache(maxsize=200000)
def resnik(a, b):
    common = ancestors_of(a).intersection(ancestors_of(b))
    if not common:
        return 0.0
    return max(ic.get(x, 0.0) for x in common)

def gene_hpo_score(gene):
    if not case_hpo:
        return 0.0
    terms = gene_terms.get(gene, set())
    if not terms:
        return 0.0
    # Case -> gene directional best-match average.
    vals = []
    for ct in case_hpo:
        vals.append(max((resnik(ct, gt) for gt in terms), default=0.0))
    raw = float(np.mean(vals)) if vals else 0.0
    return raw / MAX_IC if MAX_IC > 0 else 0.0

candidate_genes = sorted(set(g for g in variants["gene"] if g))
gene_hpo = {g: gene_hpo_score(g) for g in tqdm(candidate_genes, desc="HPO gene scores")}

MVA_GENES = {
    "BUB1B": 1.0,
    "CEP57": 1.0,
    "TRIP13": 1.0,
    "CENATAC": 1.0,
    "SLF2": 0.95,
    "SMC5": 0.95,
    "MAD1L1": 0.95,
}

variants["hpo_gene_score"] = variants["gene"].map(gene_hpo).fillna(0.0)
variants["mva_gene_prior"] = variants["gene"].map(MVA_GENES).fillna(0.0)

gene_table = pd.DataFrame({
    "gene": candidate_genes,
    "hpo_gene_score": [gene_hpo.get(g,0.0) for g in candidate_genes],
    "mva_gene_prior": [MVA_GENES.get(g,0.0) for g in candidate_genes],
}).sort_values(["mva_gene_prior","hpo_gene_score"], ascending=False)

gene_table.to_csv(PRIVATE_OUT/"gene_priority_private.csv", index=False)
gene_table.head(25)

## 6. Variant evidence score

To jest **ranking evidence score**, nie kliniczne prawdopodobieństwo patogenności.

Rzadkość:
- gnomAD absent / ultra-rare: najwyższy priorytet;
- ≤1e-3: nadal mocne;
- ≤1e-2: możliwe dla AR, ale słabsze;
- >1%: silna kara.

Nie odrzucamy automatycznie non-PASS — obniżamy score, aby nie przeoczyć prawdziwego allelu przy nietypowej jakości.

In [ ]:
HIGH = {
    "transcript_ablation","splice_acceptor_variant","splice_donor_variant",
    "stop_gained","frameshift_variant","stop_lost","start_lost"
}
MODERATE = {
    "missense_variant","inframe_insertion","inframe_deletion",
    "protein_altering_variant"
}
LOW_INTEREST = {
    "splice_region_variant","start_retained_variant","stop_retained_variant",
    "synonymous_variant"
}

def consequence_score(s):
    terms = set(str(s).split("&"))
    if terms & HIGH:
        return 1.0
    if terms & MODERATE:
        return 0.72
    if terms & LOW_INTEREST:
        return 0.35
    if "intron_variant" in terms:
        return 0.08
    return 0.02

def rarity_score(row):
    vals = [
        row.get("gnomade_af", np.nan),
        row.get("gnomadg_af", np.nan),
        row.get("max_af", np.nan),
    ]
    vals = [float(x) for x in vals if pd.notna(x)]
    if not vals:
        return 1.0
    af = max(vals)
    if af <= 1e-5:
        return 1.0
    if af <= 1e-4:
        return 0.95
    if af <= 1e-3:
        return 0.80
    if af <= 5e-3:
        return 0.58
    if af <= 1e-2:
        return 0.38
    if af <= 2e-2:
        return 0.15
    return -0.80

def clinvar_score(s):
    s = str(s).lower().replace("_"," ")
    if "pathogenic" in s and "conflicting" not in s:
        if "likely pathogenic" in s:
            return 0.85
        return 1.0
    if "conflicting" in s:
        return 0.10
    if "uncertain" in s:
        return 0.0
    if "benign" in s:
        return -1.0
    return 0.0

def predictor_score(row):
    x = 0.0
    sift = str(row.get("sift","")).lower()
    pp = str(row.get("polyphen","")).lower()
    if "deleterious" in sift:
        x += 0.5
    if "probably_damaging" in pp:
        x += 0.5
    elif "possibly_damaging" in pp:
        x += 0.25
    return min(x, 1.0)

def quality_score(row):
    s = 0.0
    if str(row.get("filter","PASS")) == "PASS":
        s += 0.45
    gq = row.get("gq", np.nan)
    dp = row.get("dp", np.nan)
    ab = row.get("allele_balance", np.nan)
    if pd.notna(gq):
        s += 0.25 if gq >= 30 else (0.12 if gq >= 20 else -0.15)
    if pd.notna(dp):
        s += 0.20 if dp >= 12 else (0.08 if dp >= 8 else -0.15)
    if row.get("zygosity") == "het" and pd.notna(ab):
        if 0.25 <= ab <= 0.75:
            s += 0.10
        elif ab < 0.15 or ab > 0.85:
            s -= 0.20
    return max(-0.5, min(1.0, s))

variants["consequence_score"] = variants["consequence"].map(consequence_score)
variants["rarity_score"] = variants.apply(rarity_score, axis=1)
variants["clinvar_score"] = variants["clin_sig"].map(clinvar_score)
variants["predictor_score"] = variants.apply(predictor_score, axis=1)
variants["quality_score"] = variants.apply(quality_score, axis=1)

variants["variant_evidence_score"] = (
    1.55*variants["consequence_score"] +
    1.40*variants["rarity_score"] +
    0.85*variants["clinvar_score"] +
    0.35*variants["predictor_score"] +
    0.35*variants["quality_score"]
)

variants["gene_context_score"] = (
    1.60*variants["hpo_gene_score"] +
    1.35*variants["mva_gene_prior"]
)

variants["single_total_score"] = (
    variants["variant_evidence_score"] +
    variants["gene_context_score"]
)

variants.to_parquet(PRIVATE_OUT/"variant_scored_private.parquet", index=False)

variants.sort_values("single_total_score", ascending=False)[[
    "gene","chrom","pos","ref","alt","zygosity","consequence","impact",
    "gnomade_af","gnomadg_af","max_af","clin_sig",
    "hpo_gene_score","mva_gene_prior","single_total_score"
]].head(30)

## 7. Compound-heterozygous pair engine

Primary pairing rules:
- dwa różne heterozygotyczne allele,
- ten sam gene symbol,
- każdy allel musi być co najmniej możliwy biologicznie,
- `MAX_AF <= 0.02` lub brak danych;
- dla znanych genów MVA dopuszczamy rzadkie introniczne warianty jako low-priority safety net.

Faza:
- ten sam `PID` + allele na przeciwnych haplotypach → `trans_supported`;
- ten sam `PID` + ten sam haplotyp → `cis_supported`;
- brak rozstrzygnięcia → `phase_unknown`.

`phase_unknown` nie jest automatycznie wadą dyskwalifikującą: warianty mogą leżeć poza jednym read-backed phase block.

In [ ]:
def pop_af(row):
    vals = [
        row.get("gnomade_af", np.nan),
        row.get("gnomadg_af", np.nan),
        row.get("max_af", np.nan),
    ]
    vals = [float(x) for x in vals if pd.notna(x)]
    return max(vals) if vals else np.nan

def plausible_for_pair(row):
    if row["zygosity"] != "het":
        return False
    af = pop_af(row)
    if pd.notna(af) and af > 0.02:
        return False

    cons = set(str(row["consequence"]).split("&"))
    if cons & (HIGH | MODERATE | LOW_INTEREST):
        return True

    # Safety net for known MVA genes: rare intronic alleles remain visible.
    if row["gene"] in MVA_GENES and "intron_variant" in cons:
        if pd.isna(af) or af <= 1e-3:
            return True
    return False

pair_pool = variants[variants.apply(plausible_for_pair, axis=1)].copy()

# Deduplicate same genomic allele/gene: keep strongest picked annotation.
pair_pool = (
    pair_pool.sort_values("single_total_score", ascending=False)
    .drop_duplicates(["variant_key","gene"])
)

# Cap per gene before combinatorics.
reduced = []
for gene, df in pair_pool.groupby("gene"):
    reduced.append(df.nlargest(10, "single_total_score"))
pair_pool = pd.concat(reduced, ignore_index=True) if reduced else pair_pool

def phase_relation(a, b):
    pid_a, pid_b = a.get("pid"), b.get("pid")
    ha, hb = a.get("alt_haplotype"), b.get("alt_haplotype")

    if pid_a not in (None,"",".") and pid_b not in (None,"",".") and str(pid_a) == str(pid_b):
        if ha in (0,1) and hb in (0,1):
            if ha != hb:
                return "trans_supported", 0.55
            else:
                return "cis_supported", -2.50
    return "phase_unknown", 0.0

pair_rows = []

for gene, df in tqdm(pair_pool.groupby("gene"), desc="Building gene pairs"):
    recs = df.to_dict("records")
    if len(recs) < 2:
        continue

    for a, b in itertools.combinations(recs, 2):
        if a["variant_key"] == b["variant_key"]:
            continue

        phase_status, phase_bonus = phase_relation(a, b)

        # Two-hit score: weak second allele matters.
        ev1, ev2 = a["variant_evidence_score"], b["variant_evidence_score"]
        ev_pair = 0.65*((ev1+ev2)/2) + 0.35*min(ev1,ev2)

        gene_context = max(a["gene_context_score"], b["gene_context_score"])

        # bonus when at least one severe/LoF-like allele exists
        severe_bonus = 0.20 if (
            set(str(a["consequence"]).split("&")) & HIGH or
            set(str(b["consequence"]).split("&")) & HIGH
        ) else 0.0

        pair_score = ev_pair + gene_context + phase_bonus + severe_bonus

        # Deterministic ordering by genomic coordinate for output.
        va, vb = sorted(
            [a,b],
            key=lambda x: (
                int(x["chrom"]) if str(x["chrom"]).isdigit() else 99,
                int(x["pos"]),
                x["ref"], x["alt"]
            )
        )

        pair_rows.append({
            "gene": gene,
            "variant1_key": va["variant_key"],
            "variant2_key": vb["variant_key"],
            "chrom_1_raw": va["chrom"],
            "pos_1": va["pos"],
            "ref_1": va["ref"],
            "alt_1": va["alt"],
            "chrom_2_raw": vb["chrom"],
            "pos_2": vb["pos"],
            "ref_2": vb["ref"],
            "alt_2": vb["alt"],
            "consequence_1": va["consequence"],
            "consequence_2": vb["consequence"],
            "impact_1": va["impact"],
            "impact_2": vb["impact"],
            "max_pop_af_1": pop_af(va),
            "max_pop_af_2": pop_af(vb),
            "clin_sig_1": va["clin_sig"],
            "clin_sig_2": vb["clin_sig"],
            "sift_1": va["sift"],
            "sift_2": vb["sift"],
            "polyphen_1": va["polyphen"],
            "polyphen_2": vb["polyphen"],
            "gq_1": va["gq"],
            "gq_2": vb["gq"],
            "dp_1": va["dp"],
            "dp_2": vb["dp"],
            "ab_1": va["allele_balance"],
            "ab_2": vb["allele_balance"],
            "pid_1_private": va["pid"],
            "pid_2_private": vb["pid"],
            "pgt_1_private": va["pgt"],
            "pgt_2_private": vb["pgt"],
            "phase_status": phase_status,
            "hpo_gene_score": max(va["hpo_gene_score"], vb["hpo_gene_score"]),
            "mva_gene_prior": max(va["mva_gene_prior"], vb["mva_gene_prior"]),
            "variant_evidence_1": ev1,
            "variant_evidence_2": ev2,
            "pair_score": pair_score,
        })

pairs = pd.DataFrame(pair_rows)
if pairs.empty:
    raise RuntimeError("No plausible heterozygous same-gene pairs found.")

pairs = pairs.sort_values(
    ["pair_score","mva_gene_prior","hpo_gene_score"],
    ascending=False
).reset_index(drop=True)

pairs.to_parquet(PRIVATE_OUT/"compound_het_pairs_private.parquet", index=False)

print("Plausible variants in pair pool:", len(pair_pool))
print("Candidate pairs:", len(pairs))
pairs[[
    "gene","variant1_key","variant2_key",
    "consequence_1","consequence_2","phase_status",
    "hpo_gene_score","mva_gene_prior","pair_score"
]].head(30)

## 8. Falsification / sensitivity views

Przed wyborem Track 1 sprawdzamy, czy top pair:
- pozostaje wysoko **bez ręcznego MVA prior**;
- pozostaje wysoko przy silniejszej karze za common variants;
- nie jest wspierany wyłącznie przez jeden słaby allel;
- nie ma dowodu `cis_supported`.

To zapobiega „odnalezieniu BUB1B tylko dlatego, że wpisaliśmy BUB1B”.

In [ ]:
# Re-score without curated MVA prior.
pairs["pair_score_no_mva_prior"] = (
    pairs["pair_score"] - 1.35*pairs["mva_gene_prior"]
)

pairs["rank_primary"] = pairs["pair_score"].rank(method="min", ascending=False).astype(int)
pairs["rank_no_mva_prior"] = pairs["pair_score_no_mva_prior"].rank(method="min", ascending=False).astype(int)
pairs["rank_shift_no_mva"] = pairs["rank_no_mva_prior"] - pairs["rank_primary"]

sensitivity = pairs[[
    "gene","variant1_key","variant2_key","phase_status",
    "rank_primary","rank_no_mva_prior","rank_shift_no_mva",
    "pair_score","pair_score_no_mva_prior",
    "hpo_gene_score","mva_gene_prior"
]].head(100)

sensitivity.to_csv(PRIVATE_OUT/"pair_sensitivity_top100.csv", index=False)
sensitivity.head(25)

## 9. Publicable Top-10 review draft

To jest **draft do naszej recenzji**, nie automatyczne submission.

Chromosome output zostaje przekształcony z wejściowego `1..22,X,Y` do formatu wymaganego przez formularz (`chr1..chr22,chrX,chrY`).

EPCR jest monotoniczną transformacją wewnętrznego score, aby spełnić schemat pliku. Nie należy interpretować go jako zwalidowanego klinicznie prawdopodobieństwa.

In [ ]:
def to_chr(x):
    x = str(x)
    if x.startswith("chr"):
        return x
    if x in {"MT","M"}:
        return "chrM"
    return "chr" + x

top10 = pairs.head(10).copy()

# Ranking-derived EPCR: preserve ordering, avoid false precision.
if len(top10) == 1:
    epcr = np.array([0.95])
else:
    scores = top10["pair_score"].to_numpy(dtype=float)
    # robust sigmoid around top-10 distribution
    med = np.median(scores)
    mad = np.median(np.abs(scores-med))
    scale = max(mad*1.4826, 0.25)
    z = (scores-med)/scale
    raw = 1/(1+np.exp(-z))
    # bound to [0.10, 0.98], then enforce strictly descending by rank
    vals = 0.10 + 0.88*raw
    vals = np.maximum.accumulate(vals[::-1])[::-1]
    for i in range(1, len(vals)):
        if vals[i] >= vals[i-1]:
            vals[i] = max(0.01, vals[i-1]-0.01)
    epcr = vals

submission = pd.DataFrame({
    "proband_id": "PROBAND01",
    "chrom_1": top10["chrom_1_raw"].map(to_chr),
    "pos_1": top10["pos_1"].astype(int),
    "ref_1": top10["ref_1"],
    "alt_1": top10["alt_1"],
    "chrom_2": top10["chrom_2_raw"].map(to_chr),
    "pos_2": top10["pos_2"].astype(int),
    "ref_2": top10["ref_2"],
    "alt_2": top10["alt_2"],
    "epcr": np.round(epcr, 4),
    "finding_type": "primary",
    "notes": [
        (
            f"gene={r.gene}; phase={r.phase_status}; "
            f"cons={r.consequence_1}+{r.consequence_2}; "
            f"HPO={r.hpo_gene_score:.3f}; MVA_prior={r.mva_gene_prior:.2f}; "
            f"internal_pair_score={r.pair_score:.3f}"
        )
        for r in top10.itertuples()
    ],
})

draft_path = PUBLIC_OUT / "NB01B_TRACK1_DRAFT_NOT_FOR_SUBMISSION.csv"
submission.to_csv(draft_path, index=False)

evidence_cols = [
    "gene","variant1_key","variant2_key","consequence_1","consequence_2",
    "impact_1","impact_2","max_pop_af_1","max_pop_af_2",
    "clin_sig_1","clin_sig_2","phase_status",
    "hpo_gene_score","mva_gene_prior",
    "variant_evidence_1","variant_evidence_2",
    "pair_score","rank_primary","rank_no_mva_prior"
]
evidence = top10[evidence_cols].copy()
evidence.to_csv(
    PUBLIC_OUT/"NB01B_TOP10_EVIDENCE.csv", index=False
)

print("DRAFT:", draft_path)
display(submission)

## 10. Decision gate for NB01C / NB03

NB01B nie kończy diagnostyki mechanizmu. Po jego wyniku wybieramy następną ścieżkę:

- **clear coding/splice compound-het in MVA gene** → warianty potwierdzić technicznie, potem `NB03 Mechanochemical Variant Atlas`;
- **pair phase_unknown** → sprawdzić, czy `PGT/PID` lub read-backed evidence może rozstrzygnąć fazę;
- **jedna mocna + jedna słaba/noncoding variant** → targeted splice/regulatory analysis;
- **brak przekonującej pary** → raw-read branch: SV/CNV, missed indels, splice/regulatory, repeat/coverage audit;
- **missense/in-frame w uporządkowanej domenie/interfejsie** → uruchomić geometry/structure branch;
- **nonsense/frameshift/splice LoF** → abundance/NMD/domain-loss branch zamiast dekoracyjnego AlphaFold.

In [ ]:
# Final audit + handoff.
top = pairs.iloc[0]

handoff_out = {
    "schema": "MVA-NB01B-HANDOFF-1.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_repo_sha": handoff.get("dataset_repo_sha"),
    "hpo_count": len(case_hpo),
    "hpo_terms": hpo_public,
    "input_vcf_records": handoff["vcfs"][0]["aggregate_stats"].get("number of records"),
    "vep_release": 116,
    "population_af_source": "VEP116 cache gnomAD v4.1; input INFO/AF explicitly not used as population AF",
    "known_mva_gene_prior": sorted(MVA_GENES),
    "compound_het_prior_source": "public organizer evaluation.py",
    "n_carried_allele_gene_rows": n_carried_allele_gene_rows,
    "n_unique_carried_alleles": n_unique_carried_alleles,
    "n_represented_genes": n_represented_genes,
    "n_pair_pool_variants": int(len(pair_pool)),
    "n_candidate_pairs": int(len(pairs)),
    "top_pair": {
        "gene": str(top["gene"]),
        "variant1": str(top["variant1_key"]),
        "variant2": str(top["variant2_key"]),
        "phase_status": str(top["phase_status"]),
        "consequence_1": str(top["consequence_1"]),
        "consequence_2": str(top["consequence_2"]),
        "hpo_gene_score": float(top["hpo_gene_score"]),
        "mva_gene_prior": float(top["mva_gene_prior"]),
        "pair_score": float(top["pair_score"]),
        "rank_no_mva_prior": int(top["rank_no_mva_prior"]),
    },
    "privacy": {
        "external_variant_api_used": False,
        "raw_genome_exported": False,
        "full_annotated_variant_table_public": False,
        "public_outputs_limited_to_hpo_and_ranked_candidate_list": True,
    },
    "outputs": {
        "private_dir": str(PRIVATE_OUT),
        "track1_draft": str(PUBLIC_OUT/"NB01B_TRACK1_DRAFT_NOT_FOR_SUBMISSION.csv"),
        "top10_evidence": str(PUBLIC_OUT/"NB01B_TOP10_EVIDENCE.csv"),
    },
}

handoff_path = PUBLIC_OUT / "NB01B_HANDOFF.json"
handoff_path.write_text(json.dumps(handoff_out, indent=2), encoding="utf-8")

print(json.dumps(handoff_out, indent=2))
print("\nNB01B COMPLETE")
print("Następny artefakt do wspólnej analizy:", handoff_path)
print("Nie przesyłaj pełnych private parquet/VCF.")